In [2]:
import numpy as np
import pandas as pd
import mne
import os
import glob

In [ ]:
""" 
Load data
"""
# Big list
ep_list = []
# # Sup grouped list
epoch_tmin = -0.1
epoch_tmax = 0.46

cleaned_data_dir = '/Users/mvmigem/Documents/data/project_2/preprocessed/mastoid-epo/'
dir_list = glob.glob(cleaned_data_dir+'*-epo.fif')

# Iterate throug data folders 
for i,sub_path in enumerate(dir_list):
    sub = int(sub_path.split('paired-mastoid-sub-')[1].split('-epo.fif')[0])
    clean_epoch_path = sub_path
    epochs = mne.read_epochs(clean_epoch_path, verbose=False)
    epochs.info['bads']= []
    ep_list.append(epochs.crop(tmin=epoch_tmin,tmax=epoch_tmax, verbose = False))
    print(epochs.info['nchan'])

# Concat list into big epoch object with all data
eps = mne.concatenate_epochs(ep_list)

In [ ]:
""" 
Subselections
"""
# Subject list
sub_list = eps.metadata['participant'].unique() 

# Add events to metadata
metadata = eps.metadata
events = eps.events

metadata['events'] = events[:,2]

eps.metadata = metadata

In [ ]:
"""
Position based ERPs
"""
evoked_pos1_list = []
evoked_pos2_list = []
evoked_pos3_list = []
evoked_pos4_list = []
for i in ep_list:
    evoked_pos1_list.append(i['pos1'].average())
    evoked_pos2_list.append(i['pos2'].average())
    evoked_pos3_list.append(i['pos3'].average())
    evoked_pos4_list.append(i['pos4'].average())

evoked_pos1 = mne.grand_average(evoked_pos1_list)
evoked_pos2 = mne.grand_average(evoked_pos2_list)
evoked_pos3 = mne.grand_average(evoked_pos3_list)
evoked_pos4 = mne.grand_average(evoked_pos4_list)

evokeds_list = [evoked_pos1,evoked_pos2,evoked_pos3,evoked_pos4]
conds = ('pos1','pos2','pos3','pos4')
# conds = ('seq1','seq2','seq3','seq4')

norm = dict(zip(conds, evokeds_list))

In [ ]:
mne.viz.plot_compare_evokeds(norm, picks="POz", vlines=[0.05,0.1])